# Feature Engineering

## Team Selection

Choose the team, Grand Prix, and season that have already been processed. These parameters define the exact comparison window for the analysis.

In [1]:
team = "Aston Martin"
grand_prix = "Japanese Grand Prix"
year = 2025

## Imports and Configuration

Load dependencies, configure the notebook path, and prepare the modules used in the exploration steps.

In [2]:
import sys
import numpy as np
from pathlib import Path
import fastf1

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src import load_data, preprocess_data, explore_data, feature_engineering

## Load Processed Data

Read the cached processed laps for both teammates so the notebook can work from already cleaned telemetry.

In [3]:
session = fastf1.get_session(year, grand_prix, "Q")
session.load()

driver_1, driver_2 = load_data.get_drivers(session, team)
segment = load_data.get_segment(session, driver_1, driver_2)
turns = load_data.get_turns(session)

try:
    lap_1, lap_2 = preprocess_data.load_processed_data(driver_1, driver_2, year, grand_prix, segment)
except Exception as e:
    print(f"Processed data for {year} {grand_prix} {driver_1} and {driver_2} not found: {e}")

req         WARNING 	DEFAULT CACHE ENABLED! (598.78 MB) C:\Users\joseg\AppData\Local\Temp\fastf1
core           INFO 	Loading data for Japanese Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '81', '16', '63', '12', '6', '44', '23', '87', '10', '55', '14', '30', '22', '27', '5', '31', '7', '18']



Processed data for 2025 Japanese Grand Prix ALO and STR found in cache.
Driver 1: ALO, Driver 2: STR, Segment: Q1


## Compute Braking features

In [4]:
try:
    braking_zones_1 = feature_engineering.analyze_braking_zones(lap_1["Interpolated"])
    braking_zones_2 = feature_engineering.analyze_braking_zones(lap_2["Interpolated"])
except Exception as e:
    print(f"Braking zones analysis failed: {e}")

try:
    braking_zones_1 = feature_engineering.assign_braking_zones_to_turns(braking_zones_1, turns)
    braking_zones_2 = feature_engineering.assign_braking_zones_to_turns(braking_zones_2, turns)
except Exception as e:
    print(f"Assignment of braking zones to turns failed: {e}")
    
print(f"Braking zones for {driver_1}:\n\n: {braking_zones_1}\n")
print(f"Braking zones for {driver_2}:\n\n: {braking_zones_2}")

print(braking_zones_1.info())

Braking zones for ALO:

:    Turn  StartDistance  EndDistance  BrakingDuration  BrakingDistance  \
0     1     679.970542   782.977661         1.512405       103.007119   
1     6    1469.691790  1512.611423         0.713592        42.919633   
2     8    2233.661258  2285.164818         0.719429        51.503560   
3     9    2345.252304  2413.923717         1.426009        68.671413   
4    11    2791.616487  2877.455754         2.424912        85.839266   
5    13    3718.680561  3787.351974         0.976355        68.671413   
6    14    3856.023387  3924.694800         1.241569        68.671413   
7    16    5255.203424  5341.042690         2.124844        85.839266   

   EntrySpeed  MinimumSpeed  
0  312.018048    204.135603  
1  239.000000    208.819108  
2  282.821207    262.830452  
3  249.141365    142.311138  
4  260.800903     79.115621  
5  282.831508    238.537660  
6  225.757158    180.807940  
7  267.099759     98.821877  

Braking zones for STR:

:     Turn  StartDist

## Compute Acceleration features

In [5]:
try:
    acceleration_zones_1 = feature_engineering.analyze_acceleration_zones(lap_1["Interpolated"], braking_zones_1)
    acceleration_zones_2 = feature_engineering.analyze_acceleration_zones(lap_2["Interpolated"], braking_zones_2)
except Exception as e:
    print(f"Acceleration zones analysis failed: {e}")
    
try:
    acceleration_zones_1 = feature_engineering.analyze_throttle_lifts(lap_1["Interpolated"], acceleration_zones_1)
    acceleration_zones_2 = feature_engineering.analyze_throttle_lifts(lap_2["Interpolated"], acceleration_zones_2)
except Exception as e:
    print(f"Throttle lifts analysis failed: {e}")

print(f"Acceleration zones for {driver_1}:\n\n: {acceleration_zones_1}\n")

Acceleration zones for ALO:

:    Turn  StartDistance  EndDistance  AccelerationDuration  \
0     1     851.649074  1461.107863              9.580394   
1     6    1572.698909  2225.077331              9.138620   
2     8    2336.668377  2336.668377              0.000000   
3     9    2431.091570  2783.032561              5.745749   
4    11    2894.623607  3710.096635             12.669809   
5    13    3813.103754  3847.439460              0.536059   
6    14    3950.446580  5246.619497             16.812494   
7    16    5358.210543  5761.655094              7.845785   

   AccelerationDistance   ExitSpeed  TimeToFullThrottle  \
0            609.458789  175.000000            0.678957   
1            652.378422  192.376692            0.466285   
2              0.000000  250.313258            0.000000   
3            351.940991  131.076242            0.837551   
4            815.473028   71.959831            2.087805   
5             34.335706  227.644628                 NaN   
6     